# Fig 4C, 4D, S9B, S9D (v2): AFR + EUR meta-analysis

This version revises the AFR and EUR simulation in `13`.

- **Study LD.** In `13`, R partially matched `dats$afr_Xmat` to the full panel. Here `gen_insample` and `susiex_insample` use LD from the sampled individuals.
- **Unrelated AFR panel.** The 1000G AFR files (PLINK 2 resources, 3,202-sample set) hold all 893 AFR, including trio children and other relatives. Cell 2 keeps a maximal unrelated set (KING kinship ≤ 0.0884; 681 AFR) and writes `1000G_AFR_unrelated_hg38/` with its own `ld_meta_file.tsv`; cell 5 further restricts it to the phase 3 unrelated set (2,504 samples, 661 AFR). The AFR reference LD and `R_finite` use all unrelated AFR; EUR uses the 1,000-individual UKB pool of `13`.
- **Total n = 600** (fits the unrelated AFR panel at AFR share 100%); NCP = 20, 40, 80, 120 as before, so h2g = NCP / 600. Folders: `ncp_<NCP>/causal_<k>/prop_<AFR share>`.
- **SuSiE-rRSS fits added** (`*_rrss`, `R_finite` = panel size, `eb_mix`), and SuSiEx runs in the same job (in-sample and reference-panel LD).
- Heterogeneous design (Fig S9D, `[afr_eur_hetero_v2]`): the `13` `[EUR_AFR_hetero]` + `[fine_prop]` pair in one job, at n = 600, NCP = 20, 30, 40.
- AFR genotype columns are standardized like the EUR ones; variants monomorphic in a sampled cohort are left out (small AFR cohorts lose rare variants), and a simulation whose causal variant is monomorphic in a cohort is dropped.
- **Identity-LD fits and saved inputs** as in `32`: `identity` / `identity_rrss`, `ld_diff` in every output, the panel LD of each region in `WORK_DIR/panel_LD/EUR_AFR_<region>.panelLD.rds`, and each cohort's in-sample LD once per region / causal / prop in `WORK_DIR/insample_LD/`.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_meta_v2/` and `AFR_EUR_hetero_v2/`. The summary uses `[meta_v2_summary]` from `32`.

Notebook `28` reads the earlier simulations and LD cache from `13`.

## Unrelated 1000G AFR panel

In [ ]:
#!/bin/bash
# Unrelated 1000G AFR panel: KING kinship on LD-pruned common variants of the AFR filesets that
# ld_meta_file.tsv points to (afr_hg38_chr1..22: all 893 AFR of the 3,202-sample set, which includes
# the trio children and other relatives), drop one of each pair of 2nd-degree-or-closer relatives
# (kinship > 0.0884), then rewrite those filesets with the unrelated samples only, next to a copy of
# ld_meta_file.tsv. Other files in the folder (all_hg38, the partial afr_hg38.pgen) are not used.
src="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_hg38"
dst="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_unrelated_hg38"
mkdir -p ${dst}/{king,log}

cat << 'EOF' > ${dst}/afr_unrelated.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J afr_unrelated
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=8000]
#BSUB -W 12:00
#BSUB -o DST/log/afr_unrelated.%J.out
#BSUB -e DST/log/afr_unrelated.%J.err
module load plink2          # any plink2 with --pmerge-list (alpha 3+); edit to your setup
set -euo pipefail
src="SRC"; dst="DST"; cd ${dst}

# filesets listed in ld_meta_file.tsv (path column, relative to src), each a complete pgen set
awk '!/^#/ && NF >= 4 {print $4}' ${src}/ld_meta_file.tsv | sort -u > prefixes.txt
while read rel; do
  for ext in pgen pvar psam; do
    [ -f ${src}/${rel}.${ext} ] || { echo "missing ${src}/${rel}.${ext}"; exit 1; }
  done
done < prefixes.txt
echo "$(wc -l < prefixes.txt) filesets from ld_meta_file.tsv"

# 1. LD-pruned common SNPs per fileset, merged genome-wide
ids="--set-all-var-ids chr@:#:\$r:\$a --new-id-max-allele-len 1000 missing --rm-dup force-first"
: > king/merge_list.txt
while read rel; do
  plink2 --pfile ${src}/${rel} ${ids} --maf 0.05 --geno 0.02 --snps-only --max-alleles 2 \
         --indep-pairwise 500kb 0.2 --threads 4 --out king/${rel}
  plink2 --pfile ${src}/${rel} ${ids} --extract king/${rel}.prune.in --make-pgen --threads 4 --out king/${rel}.pruned
  echo king/${rel}.pruned >> king/merge_list.txt
done < prefixes.txt
plink2 --pmerge-list king/merge_list.txt pfile --make-pgen --threads 4 --out king/afr_pruned

# 2. KING-robust kinship; keep a maximal set with no pair above 2nd degree
plink2 --pfile king/afr_pruned --king-cutoff 0.0884 --threads 4 --out king/afr_king
cp king/afr_king.king.cutoff.in.id unrelated.id
echo "AFR samples: $(($(wc -l < king/afr_pruned.psam) - 1)); unrelated: $(($(wc -l < unrelated.id) - 1))"

# 3. each fileset restricted to the unrelated samples (same names)
while read rel; do
  plink2 --pfile ${src}/${rel} --keep unrelated.id --make-pgen --threads 4 --out ${dst}/${rel}
done < prefixes.txt

# 4. ld_meta_file for the new files (its paths are relative to the meta file, so it is copied as is)
cp ${src}/ld_meta_file.tsv ${dst}/ld_meta_file.tsv
EOF
sed -i -e "s|SRC|${src}|g" -e "s|DST|${dst}|g" ${dst}/afr_unrelated.sh
bsub < ${dst}/afr_unrelated.sh

In [ ]:
# checks: unrelated count (total_n must not exceed it) and the new meta file
dst=/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_unrelated_hg38
tail -2 ${dst}/log/afr_unrelated.*.out
wc -l < ${dst}/unrelated.id      # unrelated + header
head -3 ${dst}/ld_meta_file.tsv

To restrict the panel further, run this step after the check above. It intersects the 681 AFR participants from the KING filter with the 661 AFR participants in the published Phase 3 unrelated set and rewrites the AFR filesets in place. The KING cutoff excludes pairs related to the second degree or closer, and the Phase 3 list excludes all documented pedigree relatives.

In [ ]:
# Restrict the unrelated AFR panel to the 1000G phase 3 unrelated set (2,504 samples; 661 AFR):
# keep = KING-unrelated (cell 2) ∩ phase 3 AFR, then rewrite the AFR filesets with it.
# Rerunning cell 2 afterwards would overwrite unrelated.id with the KING-only list again.
dst="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_unrelated_hg38"
src="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_hg38"
cd ${dst}

wget -q -O phase3_2504.panel \
  http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel
awk 'NR > 1 && $3 == "AFR" {print $1}' phase3_2504.panel | sort -u > phase3_afr.txt
echo "phase 3 samples: $(($(wc -l < phase3_2504.panel) - 1)); AFR: $(wc -l < phase3_afr.txt)"

# KING list kept as unrelated_king.id; unrelated.id becomes the intersection (header kept)
[ -f unrelated_king.id ] || cp unrelated.id unrelated_king.id
awk 'NR == FNR {p[$1] = 1; next} FNR == 1 || ($NF in p)' phase3_afr.txt unrelated_king.id > unrelated.id
echo "KING-unrelated: $(($(wc -l < unrelated_king.id) - 1)); also in phase 3: $(($(wc -l < unrelated.id) - 1))"
echo "phase 3 AFR removed by KING: $(awk 'NR == FNR {k[$NF] = 1; next} !($1 in k)' unrelated_king.id phase3_afr.txt | wc -l)"

cat << 'JOB' > afr_phase3.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q express
#BSUB -J afr_phase3
#BSUB -n 2
#BSUB -R rusage[mem=8000]
#BSUB -W 2:00
#BSUB -o DST/log/afr_phase3.%J.out
#BSUB -e DST/log/afr_phase3.%J.err
module load plink2
set -euo pipefail
while read rel; do
  plink2 --pfile SRC/${rel} --keep DST/unrelated.id --make-pgen --threads 2 --out DST/${rel}
done < DST/prefixes.txt
echo "samples per fileset: $(($(wc -l < DST/afr_hg38_chr1.psam) - 1))"
JOB
sed -i -e "s|SRC|${src}|g" -e "s|DST|${dst}|g" afr_phase3.sh
bsub < afr_phase3.sh

## Simulate and fine-map

In [ ]:
[afr_eur_meta_v2]
# AFR + EUR meta-analysis (v2): simulate both cohorts and fine-map in one job.
#   AFR cohort: unrelated 1000G AFR individuals; reference = all of them.
#   EUR cohort: the 1,000-individual UKB pool (13 cell 2); reference = the whole pool.
#   In-sample LD = LD of the sampled individuals, mixed by w = n_AFR / N (exact, oracle).
#   Reference LD = each cohort's full panel, single or mixed (true or estimated weights).
#   Identity LD (SuSiE-RSS and SuSiE-rRSS with R_finite = B_identity) as a no-LD baseline.
# Per-SNP h2g = ncp / total_n. SuSiEx (per-population z and LD) runs when susiex_bin is set.
# Inputs kept for later checks: z and sampled rows in every output; the panel LD of each region
# once in panel_ld_dir; each cohort's in-sample LD once per region / causal / prop in insample_ld_dir.
# ld_diff: MSE / max |dLD| / correlation of every reference LD (and identity) against the in-sample LD.
parameter: genofile = paths
# 1000G EUR bim files (kg_trimed_random), used for the region index (as in 13)
parameter: cwd = path("output")
parameter: eur_pool_dir = "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_1k/result"
parameter: afr_ld_meta = "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_unrelated_hg38/ld_meta_file.tsv"
parameter: susiex_bin = "/sc/arion/projects/ad-omics/haochen/rss_qc/susie_x/SuSiEx/bin/SuSiEx"
parameter: n_causal = 1
parameter: ncp = 40
parameter: total_n = 600
parameter: afr_prop = 0.5
# AFR sample-size proportion: n_AFR = round(afr_prop * total_n), n_EUR = total_n - n_AFR
parameter: B_identity = 10000
# R_finite of the identity-LD SuSiE-rRSS fit (the largest B of 28 [cross_ref_B_sweep])
parameter: panel_ld_dir = ""
# per-region panel LD cache (written once per region if missing); "" = not saved
parameter: insample_ld_dir = ""
# in-sample LD cache, one file per region / causal / prop (shared by the ncp runs); "" = not saved
parameter: ld_thresh = 0.05
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "30G"
parameter: numThreads = 2
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_ncp_{ncp}_prop_{afr_prop}_meta.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(matrixStats)
    library(Rfast)

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    `%||%` <- function(a, b) if (is.null(a)) b else a

    # susieR GitHub build (multi-panel R, R_finite, eb_mix) in its own environment, as in 28
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

    out_path <- ${_output:r}
    file     <- ${_input:r}
    i        <- file %>% str_remove("\\.bim$") %>% str_remove(".*/EUR_KG_")
    n_causal <- ${n_causal}; ncp <- ${ncp}; total_n <- ${total_n}; prop <- ${afr_prop}

    save_dropped <- function(reason) {
      saveRDS(list(status = "dropped", reason = reason, region = i, index = ${_index},
                   ncp = ncp, causal = n_causal, prop = "${afr_prop}", input_path = file), out_path)
      cat(sprintf("[DROP] region %s prop %s: %s\n", i, prop, reason))
      quit(save = "no", status = 0)
    }

    ## ---- region data: EUR pool and unrelated AFR genotypes, allele-aligned (as 13) ----
    eur_data  <- plink_process(file.path("${eur_pool_dir}", paste0("UKBB_", i)))
    target_tb <- eur_data$bim %>%
      mutate(variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
             variant_id = variants_id_original)
    region     <- sprintf("chr%s:%d-%d", unique(target_tb$chrom), min(target_tb$pos), max(target_tb$pos))
    res_sketch <- load_LD_matrix("${afr_ld_meta}", region, return_genotype = TRUE)

    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
      idx <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
      colnames(sketch2) <- qced$variants_id_original
      list(sumstat = qced, sketch = sketch2)
    }
    harm <- sketch_harmonize(target_data = target_tb,
                             ref_variants = res_sketch$ref_panel %>% select(-variant_id),
                             sketch = res_sketch$LD_matrix)

    EUR_full <- eur_data$bed[, colnames(harm$sketch), drop = FALSE]    # scaled by plink_process
    AFR_full <- harm$sketch
    rm(harm, res_sketch, eur_data); invisible(gc())

    ## ---- simulate both cohorts ----
    # variants polymorphic in both panels; AFR columns standardized like the EUR ones so that a
    # causal variant has the same standardized effect in both cohorts
    keep0    <- colVars(EUR_full) > 0 & colVars(AFR_full) > 0
    EUR_full <- EUR_full[, keep0, drop = FALSE]
    AFR_full <- scale(AFR_full[, keep0, drop = FALSE])
    p     <- ncol(EUR_full)
    B_EUR <- nrow(EUR_full)                      # reference size of each panel (R_finite)
    B_AFR <- nrow(AFR_full)

    n_afr <- round(prop * total_n)
    n_eur <- total_n - n_afr
    if (n_eur > B_EUR) stop(sprintf("n_eur = %d exceeds the EUR pool (%d)", n_eur, B_EUR))
    if (n_afr > B_AFR) stop(sprintf("n_afr = %d exceeds the AFR panel (%d)", n_afr, B_AFR))
    w   <- n_afr / total_n
    h2g <- ncp / total_n                         # per-SNP h2g (sim_multi_traits, is_h2g_total = FALSE)

    # shared causal architecture, drawn on the EUR panel and seeded by the region (as in 13)
    set.seed(as.integer(i))
    causal_index <- sample_causal_index(Xmat = EUR_full, ncausal = n_causal)
    B_mat <- matrix(0, nrow = p, ncol = 1)
    B_mat[, 1] <- sim_beta_fix_variant(G = EUR_full, causal_index = causal_index, is_h2g_total = FALSE)

    draw <- function(G, n) {
      if (n == 0) return(list(idx = integer(0), X = G[0, , drop = FALSE], y = numeric(0)))
      idx <- sort(sample.int(nrow(G), n))
      X <- G[idx, , drop = FALSE]
      list(idx = idx, X = X, y = as.numeric(sim_multi_traits(G = X, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P[, 1]))
    }
    EUR <- draw(EUR_full, n_eur)
    AFR <- draw(AFR_full, n_afr)
    idx_eur <- EUR$idx; idx_afr <- AFR$idx      # rows of each panel used by the cohort

    # analysis variants: polymorphic in every sampled cohort (small AFR cohorts lose rare variants)
    keep <- rep(TRUE, p)
    for (co in list(EUR, AFR)) if (length(co$idx) > 1) keep <- keep & colVars(co$X) > 0
    if (!all(keep[causal_index])) save_dropped("a causal variant is monomorphic in a sampled cohort")
    ref_order <- colnames(EUR_full)[keep]
    true_vars <- match(colnames(EUR_full)[causal_index], ref_order)

    ## ---- per-cohort and natural-weight meta z ----
    calc_z <- function(co) {
      if (length(co$idx) == 0) return(NULL)
      rr <- susieR::univariate_regression(co$X[, ref_order, drop = FALSE], co$y)
      rr$betahat / rr$sebetahat
    }
    z_eur  <- calc_z(EUR)
    z_afr  <- calc_z(AFR)
    z_meta <- (sqrt(n_afr) * (z_afr %||% 0) + sqrt(n_eur) * (z_eur %||% 0)) / sqrt(total_n)

    ## ---- LD ----
    ld_of <- function(G) { R <- get_cormat(G[, ref_order, drop = FALSE]); dimnames(R) <- list(ref_order, ref_order); R }
    in_LD <- list()                              # per-population in-sample LD (also used by SuSiEx)
    if (n_eur > 0) in_LD$EUR <- ld_of(EUR$X)
    if (n_afr > 0) in_LD$AFR <- ld_of(AFR$X)
    R_gen_in   <- w * (in_LD$AFR %||% 0) + (1 - w) * (in_LD$EUR %||% 0)   # in-sample, exact (oracle)
    EUR_LD     <- ld_of(EUR_full)
    AFR_LD     <- ld_of(AFR_full)
    R_gen_full <- w * AFR_LD + (1 - w) * EUR_LD                          # reference panels, true weight
    I_p        <- diag(length(ref_order)); dimnames(I_p) <- dimnames(EUR_LD)

    # in-sample LD of each cohort. The sampled rows do not depend on ncp (same seed and draws), so
    # the ncp runs of a region / causal / prop share one file; a cohort that uses its whole panel is
    # not saved (its in-sample LD is the panel LD). Different rows under the same name (not expected)
    # get their own file.
    insample_ld_dir <- "${insample_ld_dir}"
    insample_path <- NULL
    if (nzchar(insample_ld_dir)) {
      insample_path <- file.path(insample_ld_dir, sprintf("EUR_AFR_%s_causal_%s_prop_%s.insampleLD.rds", i, n_causal, "${afr_prop}"))
      same_rows <- function(f) { old <- readRDS(f); identical(old$idx_eur, idx_eur) && identical(old$idx_afr, idx_afr) }
      if (!(file.exists(insample_path) && same_rows(insample_path))) {
        if (file.exists(insample_path))
          insample_path <- sub("\\.insampleLD\\.rds$", paste0("_ncp_", ncp, ".insampleLD.rds"), insample_path)
        dir.create(insample_ld_dir, recursive = TRUE, showWarnings = FALSE)
        full <- c(EUR = n_eur == B_EUR, AFR = n_afr == B_AFR)
        tmp  <- paste0(insample_path, ".tmp", Sys.getpid())
        saveRDS(list(region = i, causal = n_causal, prop = "${afr_prop}", w = w, ref_order = ref_order,
                     idx_eur = idx_eur, idx_afr = idx_afr, full_panel = full,
                     R_EUR_in = if (n_eur > 0 && !full[["EUR"]]) in_LD$EUR,
                     R_AFR_in = if (n_afr > 0 && !full[["AFR"]]) in_LD$AFR), tmp)
        invisible(file.rename(tmp, insample_path))
      }
    }

    # panel LD of the region over all variants polymorphic in both panels: the same for every
    # ncp / causal / prop (each run's ref_order is a subset); written once, atomically
    panel_ld_dir <- "${panel_ld_dir}"
    if (nzchar(panel_ld_dir)) {
      panel_path <- file.path(panel_ld_dir, paste0("EUR_AFR_", i, ".panelLD.rds"))
      if (!file.exists(panel_path)) {
        dir.create(panel_ld_dir, recursive = TRUE, showWarnings = FALSE)
        v <- colnames(EUR_full)
        ld_all <- function(G) { R <- get_cormat(G); dimnames(R) <- list(v, v); R }
        tmp <- paste0(panel_path, ".tmp", Sys.getpid())
        saveRDS(list(region = i, variant_id = v, EUR_LD = ld_all(EUR_full), AFR_LD = ld_all(AFR_full),
                     B_EUR = B_EUR, B_AFR = B_AFR), tmp)
        invisible(file.rename(tmp, panel_path))
      }
    }

    ## ---- fits: SuSiE-RSS and SuSiE-rRSS (R_finite = panel size, eb_mix) ----
    fit <- function(tag, ...) tryCatch(
      suppressWarnings(susie_hc(z = z_meta, n = total_n, ...)),
      error = function(e) { cat(sprintf("[FAIL-FIT] region %s prop %s %s: %s\n", i, prop, tag, conditionMessage(e))); NULL })
    rrss <- function(tag, R, B) fit(tag, R = R, R_finite = B, R_mismatch = "eb_mix")

    result_list <- list(
      gen_insample      = fit("gen_insample", R = R_gen_in),
      gen_refpanel      = fit("gen_refpanel", R = R_gen_full),
      eur_refpanel      = fit("eur_refpanel", R = EUR_LD),
      afr_refpanel      = fit("afr_refpanel", R = AFR_LD),
      mix_refpanel      = fit("mix_refpanel", R = list(AFR_LD, EUR_LD)),       # omega[1] = AFR, [2] = EUR
      eur_refpanel_rrss = rrss("eur_refpanel_rrss", EUR_LD, B_EUR),
      afr_refpanel_rrss = rrss("afr_refpanel_rrss", AFR_LD, B_AFR),
      mix_refpanel_rrss = rrss("mix_refpanel_rrss", list(AFR_LD, EUR_LD), c(B_AFR, B_EUR)),
      identity          = fit("identity", R = I_p),
      identity_rrss     = rrss("identity_rrss", I_p, ${B_identity})
    )
    om      <- result_list$mix_refpanel$omega_weights
    om_rrss <- result_list$mix_refpanel_rrss$omega_weights
    mixed   <- function(o) if (length(o) == 2) o[1] * AFR_LD + o[2] * EUR_LD else NULL
    result_list$manual_refpanel <- if (!is.null(mixed(om))) fit("manual_refpanel", R = mixed(om))

    ## ---- SuSiEx: per-population z with in-sample or reference-panel LD (13 [finemap_susiex]) ----
    run_susiex <- function(pops, snp, bin, workdir, out_name = "sx",
                           n_sig = 10, level = 0.95, min_purity = 0.5,
                           pval_thresh = 1e-5, mult_step = TRUE, threads = 1) {
      m   <- length(snp)
      bp  <- seq_len(m)                       # synthetic, unique, shared across pops
      a1  <- rep("A", m); a2 <- rep("G", m)   # dummy consistent alleles (inputs pre-harmonized)
      dir.create(workdir, showWarnings = FALSE, recursive = TRUE)
      prefixes <- character(length(pops)); ssts <- character(length(pops)); ns <- numeric(length(pops))
      for (k in seq_along(pops)) {
        nm <- names(pops)[k]; pk <- pops[[k]]
        z  <- as.numeric(pk$z); R <- as.matrix(pk$R); diag(R) <- 1
        pref <- file.path(workdir, nm); prefixes[k] <- pref; ns[k] <- pk$n
        sst <- data.frame(chr = "1", snp = snp, bp = bp, A1 = a1, A2 = a2,
                          BETA = z, SE = 1, P = 2 * pnorm(-abs(z)), stringsAsFactors = FALSE)
        ssts[k] <- paste0(pref, ".sumstats.txt")
        write.table(sst, ssts[k], sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
        write.table(data.frame("1", snp, 0, bp, a2, a1), paste0(pref, "_ref.bim"),
                    sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)   # col5=a2,col6=a1
        write.table(data.frame(CHR = "1", SNP = snp, A1 = a2, A2 = a1, MAF = 0.3,
                               NCHROBS = round(2 * pk$n)), paste0(pref, "_frq.frq"),
                    sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)     # col3=a2,col4=a1
        con <- file(paste0(pref, ".ld.bin"), "wb"); writeBin(as.numeric(as.vector(R)), con, size = 4); close(con)
      }
      cc <- function(v) paste(rep(v, length(pops)), collapse = ",")
      args <- c(
        paste0("--sst_file=", paste(ssts, collapse = ",")),
        paste0("--n_gwas=",   paste(ns, collapse = ",")),
        paste0("--ld_file=",  paste(prefixes, collapse = ",")),
        paste0("--out_dir=",  workdir), paste0("--out_name=", out_name),
        "--chr=1", paste0("--bp=", min(bp), ",", max(bp)),
        paste0("--chr_col=", cc(1)), paste0("--snp_col=", cc(2)), paste0("--bp_col=", cc(3)),
        paste0("--a1_col=", cc(4)), paste0("--a2_col=", cc(5)), paste0("--eff_col=", cc(6)),
        paste0("--se_col=", cc(7)), paste0("--pval_col=", cc(8)),
        paste0("--n_sig=", n_sig), paste0("--level=", level),
        paste0("--min_purity=", min_purity), paste0("--pval_thresh=", pval_thresh),
        "--maf=0", paste0("--mult-step=", ifelse(mult_step, "True", "False")),
        "--keep-ambig=True", "--plink=plink", paste0("--threads=", threads))
      log <- suppressWarnings(system2(bin, args, stdout = TRUE, stderr = TRUE))

      # ---- parse .cs -> credible sets as INDICES into `snp` (= ref_order) ----
      cs_path <- file.path(workdir, paste0(out_name, ".cs"))
      sets <- list()
      if (file.exists(cs_path)) {
        raw <- readLines(cs_path); raw <- raw[!grepl("^#", raw)]; raw <- raw[nzchar(trimws(raw))]
        if (length(raw) && !grepl("^(NULL|FAIL)", raw[1])) {
          cs <- read.table(text = paste(raw, collapse = "\n"), header = TRUE, sep = "\t",
                           check.names = FALSE, stringsAsFactors = FALSE)
          sets <- lapply(split(cs$SNP, cs$CS_ID), function(ids) match(as.character(ids), snp))
          names(sets) <- paste0("L", names(sets))
        }
      }
      # ---- parse .snp -> overall PIP over `snp` ----
      pip <- setNames(numeric(m), snp)
      snp_path <- file.path(workdir, paste0(out_name, ".snp"))
      if (file.exists(snp_path)) {
        sp <- try(read.table(snp_path, header = TRUE, sep = "\t", check.names = FALSE,
                             stringsAsFactors = FALSE), silent = TRUE)
        if (!inherits(sp, "try-error") && "SNP" %in% names(sp)) {
          pcols <- grep("^PIP\\(CS", names(sp), value = TRUE)
          if (length(pcols)) {
            ov <- 1 - apply(1 - as.matrix(sp[, pcols, drop = FALSE]), 1, prod)
            pip[match(as.character(sp$SNP), snp)] <- ov
          }
        }
      }
      list(sets = list(cs = sets), pip = unname(pip), n_cs = length(sets), log = log)
    }

    susiex_bin <- "${susiex_bin}"
    if (nzchar(susiex_bin) && file.exists(susiex_bin)) {
      zz   <- list(EUR = z_eur, AFR = z_afr); zz <- zz[!vapply(zz, is.null, logical(1))]
      nn   <- c(EUR = n_eur, AFR = n_afr)
      pops <- function(LD) setNames(lapply(names(zz), function(k) list(z = zz[[k]], R = LD[[k]], n = nn[[k]])), names(zz))
      tmp  <- file.path(dirname(out_path), paste0(".sxtmp_", tools::file_path_sans_ext(basename(out_path))))
      sx   <- function(LD, tag) tryCatch(run_susiex(pops(LD), ref_order, susiex_bin, file.path(tmp, tag), tag),
                                         error = function(e) { cat(sprintf("[FAIL-FIT] %s: %s\n", tag, conditionMessage(e))); NULL })
      result_list$susiex_insample <- sx(in_LD, "susiex_insample")
      result_list$susiex_refpanel <- sx(list(EUR = EUR_LD, AFR = AFR_LD), "susiex_refpanel")
      unlink(tmp, recursive = TRUE)
    }

    # largest |r| between the top variants of different CSs, against the LD each fit used
    # (identity fits: against the in-sample LD, since the identity has no off-diagonal LD)
    cross_cs_maxcorr <- function(fit, R) {
      cs <- fit$sets$cs
      if (is.null(fit) || is.null(R) || length(cs) < 2) return(NA_real_)
      tops <- vapply(cs, function(m) m[which.max(fit$pip[m])], integer(1))
      cc <- abs(R[tops, tops]); diag(cc) <- NA
      max(cc, na.rm = TRUE)
    }
    R_used <- list(gen_insample = R_gen_in, gen_refpanel = R_gen_full,
                   eur_refpanel = EUR_LD, afr_refpanel = AFR_LD, mix_refpanel = mixed(om),
                   eur_refpanel_rrss = EUR_LD, afr_refpanel_rrss = AFR_LD, mix_refpanel_rrss = mixed(om_rrss),
                   manual_refpanel = mixed(om), susiex_insample = R_gen_in, susiex_refpanel = R_gen_full,
                   identity = R_gen_in, identity_rrss = R_gen_in)
    max_cs_corr <- imap_dbl(result_list, function(f, nm) if (is.null(f)) NA_real_ else cross_cs_maxcorr(f, R_used[[nm]]))

    ## ---- LD difference of each reference LD from the in-sample LD (metrics as 28 [cross_ref_ld_similarity]) ----
    ld_metrics <- function(R_ref, R_true, filter) {
      ut <- upper.tri(R_true); a <- R_ref[ut]; b <- R_true[ut]
      ok <- is.finite(a) & is.finite(b)
      if (filter) ok <- ok & abs(b) > ${ld_thresh}
      if (!any(ok)) return(tibble(mse = NA_real_, max_abs_diff = NA_real_, concord = NA_real_, n_pairs = 0L))
      d <- a[ok] - b[ok]
      tibble(mse = mean(d^2), max_abs_diff = max(abs(d)),
             concord = if (sum(ok) > 1 && sd(a[ok]) > 0) suppressWarnings(cor(a[ok], b[ok])) else NA_real_,
             n_pairs = sum(ok))
    }
    ld_refs <- compact(list(gen_refpanel = R_gen_full, eur_refpanel = EUR_LD, afr_refpanel = AFR_LD,
                            mix_refpanel = mixed(om), mix_refpanel_rrss = mixed(om_rrss), identity = I_p))
    ld_diff <- imap_dfr(ld_refs, function(R, ref)
      map_dfr(setNames(c(TRUE, FALSE), c("insample_r_gt_${ld_thresh}", "all")), function(flt) ld_metrics(R, R_gen_in, flt), .id = "pairs") %>%
        mutate(reference = ref, .before = 1))

    output <- list(
      status = "ok", design = "v2",
      result_list = result_list,
      region = i, index = ${_index}, input_path = file,
      ncp = ncp, h2g = h2g, causal = n_causal, prop = "${afr_prop}", w = w,
      n_sample = total_n, total_n = total_n, n_eur = n_eur, n_afr = n_afr,
      B_EUR = B_EUR, B_AFR = B_AFR,
      ref_order = ref_order, true_vars = true_vars,          # true_vars index ref_order, as the CSs do
      z_meta = z_meta, z_eur = z_eur, z_afr = z_afr,
      idx_eur = idx_eur, idx_afr = idx_afr,
      omega_weights = om, omega_weights_rrss = om_rrss,
      max_cs_corr = max_cs_corr, ld_diff = ld_diff,
      panel_ld_path = if (nzchar(panel_ld_dir)) panel_path, insample_ld_path = insample_path
    )
    dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output, out_path)

    ncs <- function(f) if (is.null(f)) NA else length(f$sets$cs)
    cat(sprintf("[OK] region %s prop %s ncp %s: CS in-sample=%s ref=%s EUR=%s AFR=%s mix=%s mix_rrss=%s susiex=%s; w=%.2f omega=%s\n",
                i, prop, ncp, ncs(result_list$gen_insample), ncs(result_list$gen_refpanel),
                ncs(result_list$eur_refpanel), ncs(result_list$afr_refpanel), ncs(result_list$mix_refpanel),
                ncs(result_list$mix_refpanel_rrss), ncs(result_list$susiex_refpanel), w,
                paste(round(om, 2), collapse = "/")))

In [ ]:
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="AFR_EUR_meta_v2"
job_dir="${base_dir}/${job_name}"
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_CAUSAL_NCP_PROP
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 90:00
#BSUB -o WORK_DIR/log/JOB_NAME_CAUSAL_NCP_PROP.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_CAUSAL_NCP_PROP.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/33.Fig4C_4D_S9B_S9D.AFR_EUR_Meta_v2.ipynb" afr_eur_meta_v2 \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/kg_trimed_random/*.bim) \
    --n_causal CAUSAL --ncp NCP --afr_prop PROP --total_n 600 \
    --panel_ld_dir WORK_DIR/panel_LD --insample_ld_dir WORK_DIR/insample_LD \
    --cwd WORK_DIR/result/ncp_NCP/causal_CAUSAL/prop_PROP/ \
    --mem 30G -j 4
EOF
chmod +x ${job_name}.sh

for causal in 1 2 3; do
  for ncp in 20 40 80 120; do
    for prop in 0.0 0.20 0.50 0.80 1.0; do
      output_script="${job_name}_causal_${causal}_ncp_${ncp}_prop_${prop}.sh"
      sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" \
          -e "s|CAUSAL|${causal}|g" \
          -e "s|NCP|${ncp}|g" \
          -e "s|PROP|${prop}|g" \
          ${job_name}.sh > "${output_script}"
      sb "${output_script}"
    done
  done
done

## Heterogeneous causal variants: AFR weight estimation (Fig S9D)

In [ ]:
[afr_eur_hetero_v2]
# AFR + EUR with population-specific causal variants (v2 of 13 [EUR_AFR_hetero] + [fine_prop]):
# each cohort has its own causal variants (disjoint), drawn and calibrated on its own panel.
# Only the AFR weight is estimated, from the reference-panel mixture fits (Fig S9D).
parameter: genofile = paths
# 1000G EUR bim files (kg_trimed_random), used for the region index (as in 13)
parameter: cwd = path("output")
parameter: eur_pool_dir = "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_1k/result"
parameter: afr_ld_meta = "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_unrelated_hg38/ld_meta_file.tsv"
parameter: n_causal = 1
parameter: ncp = 30
parameter: total_n = 600
parameter: afr_prop = 0.5
# AFR sample-size proportion: n_AFR = round(afr_prop * total_n), n_EUR = total_n - n_AFR
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "30G"
parameter: numThreads = 2
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_ncp_{ncp}_prop_{afr_prop}_hetero.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(matrixStats)
    library(Rfast)

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    `%||%` <- function(a, b) if (is.null(a)) b else a

    # susieR GitHub build (multi-panel R, R_finite, eb_mix) in its own environment, as in 28
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

    out_path <- ${_output:r}
    file     <- ${_input:r}
    i        <- file %>% str_remove("\\.bim$") %>% str_remove(".*/EUR_KG_")
    n_causal <- ${n_causal}; ncp <- ${ncp}; total_n <- ${total_n}; prop <- ${afr_prop}

    save_dropped <- function(reason) {
      saveRDS(list(status = "dropped", reason = reason, region = i, index = ${_index},
                   ncp = ncp, causal = n_causal, prop = "${afr_prop}", input_path = file), out_path)
      cat(sprintf("[DROP] region %s prop %s: %s\n", i, prop, reason))
      quit(save = "no", status = 0)
    }

    ## ---- region data: EUR pool and unrelated AFR genotypes, allele-aligned (as 13) ----
    eur_data  <- plink_process(file.path("${eur_pool_dir}", paste0("UKBB_", i)))
    target_tb <- eur_data$bim %>%
      mutate(variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
             variant_id = variants_id_original)
    region     <- sprintf("chr%s:%d-%d", unique(target_tb$chrom), min(target_tb$pos), max(target_tb$pos))
    res_sketch <- load_LD_matrix("${afr_ld_meta}", region, return_genotype = TRUE)

    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
      idx <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
      colnames(sketch2) <- qced$variants_id_original
      list(sumstat = qced, sketch = sketch2)
    }
    harm <- sketch_harmonize(target_data = target_tb,
                             ref_variants = res_sketch$ref_panel %>% select(-variant_id),
                             sketch = res_sketch$LD_matrix)

    EUR_full <- eur_data$bed[, colnames(harm$sketch), drop = FALSE]    # scaled by plink_process
    AFR_full <- harm$sketch
    rm(harm, res_sketch, eur_data); invisible(gc())

    ## ---- simulate both cohorts ----
    # variants polymorphic in both panels; AFR columns standardized like the EUR ones so that a
    # causal variant has the same standardized effect in both cohorts
    keep0    <- colVars(EUR_full) > 0 & colVars(AFR_full) > 0
    EUR_full <- EUR_full[, keep0, drop = FALSE]
    AFR_full <- scale(AFR_full[, keep0, drop = FALSE])
    p     <- ncol(EUR_full)
    B_EUR <- nrow(EUR_full)                      # reference size of each panel (R_finite)
    B_AFR <- nrow(AFR_full)

    n_afr <- round(prop * total_n)
    n_eur <- total_n - n_afr
    if (n_eur > B_EUR) stop(sprintf("n_eur = %d exceeds the EUR pool (%d)", n_eur, B_EUR))
    if (n_afr > B_AFR) stop(sprintf("n_afr = %d exceeds the AFR panel (%d)", n_afr, B_AFR))
    w   <- n_afr / total_n
    h2g <- ncp / total_n                         # per-SNP h2g (sim_multi_traits, is_h2g_total = FALSE)

    # population-specific causal variants (disjoint), each calibrated on its own panel (as in 13)
    set.seed(as.integer(i))
    causal_eur <- sample_causal_index(Xmat = EUR_full, ncausal = n_causal)
    repeat {
      causal_afr <- sample_causal_index(Xmat = AFR_full, ncausal = n_causal)
      if (length(intersect(causal_afr, causal_eur)) == 0) break
    }
    beta_of <- function(G, idx) { B <- matrix(0, nrow = p, ncol = 1); B[, 1] <- sim_beta_fix_variant(G = G, causal_index = idx, is_h2g_total = FALSE); B }
    B_eur <- beta_of(EUR_full, causal_eur)
    B_afr <- beta_of(AFR_full, causal_afr)

    draw <- function(G, n, B) {
      if (n == 0) return(list(idx = integer(0), X = G[0, , drop = FALSE], y = numeric(0)))
      idx <- sort(sample.int(nrow(G), n))
      X <- G[idx, , drop = FALSE]
      list(idx = idx, X = X, y = as.numeric(sim_multi_traits(G = X, B = B, h2g = h2g, is_h2g_total = FALSE)$P[, 1]))
    }
    EUR <- draw(EUR_full, n_eur, B_eur)
    AFR <- draw(AFR_full, n_afr, B_afr)

    keep <- rep(TRUE, p)
    for (co in list(EUR, AFR)) if (length(co$idx) > 1) keep <- keep & colVars(co$X) > 0
    if (n_eur > 0 && !all(keep[causal_eur])) save_dropped("an EUR causal variant is monomorphic in the EUR cohort")
    if (n_afr > 0 && !all(keep[causal_afr])) save_dropped("an AFR causal variant is monomorphic in the AFR cohort")
    ref_order <- colnames(EUR_full)[keep]

    calc_z <- function(co) {
      if (length(co$idx) == 0) return(NULL)
      rr <- susieR::univariate_regression(co$X[, ref_order, drop = FALSE], co$y)
      rr$betahat / rr$sebetahat
    }
    z_meta <- (sqrt(n_afr) * (calc_z(AFR) %||% 0) + sqrt(n_eur) * (calc_z(EUR) %||% 0)) / sqrt(total_n)

    ld_of  <- function(G) { R <- get_cormat(G[, ref_order, drop = FALSE]); dimnames(R) <- list(ref_order, ref_order); R }
    EUR_LD <- ld_of(EUR_full)
    AFR_LD <- ld_of(AFR_full)

    ## ---- mixture fits on the reference panels: estimated AFR weight ----
    fit <- function(tag, ...) tryCatch(
      suppressWarnings(susie_hc(z = z_meta, R = list(AFR_LD, EUR_LD), n = total_n, ...)),   # omega[1] = AFR
      error = function(e) { cat(sprintf("[FAIL-FIT] region %s prop %s %s: %s\n", i, prop, tag, conditionMessage(e))); NULL })
    om      <- fit("mix_refpanel")$omega_weights
    om_rrss <- fit("mix_refpanel_rrss", R_finite = c(B_AFR, B_EUR), R_mismatch = "eb_mix")$omega_weights

    output <- list(
      status = "ok", design = "v2",
      omega_weights = om, omega_weights_rrss = om_rrss,      # c(AFR, EUR)
      est_afr_prop = if (length(om) == 2) om[1] else NA_real_,
      est_afr_prop_rrss = if (length(om_rrss) == 2) om_rrss[1] else NA_real_,
      w = w,                                                  # true AFR proportion
      region = i, index = ${_index}, input_path = file,
      ncp = ncp, h2g = h2g, causal = n_causal, prop = "${afr_prop}",
      total_n = total_n, n_eur = n_eur, n_afr = n_afr, B_EUR = B_EUR, B_AFR = B_AFR,
      true_vars_eur = match(colnames(EUR_full)[causal_eur], ref_order),
      true_vars_afr = match(colnames(EUR_full)[causal_afr], ref_order)
    )
    dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output, out_path)
    cat(sprintf("[OK] region %s prop %s ncp %s: est AFR = %.3f (rRSS %.3f), true = %.3f\n",
                i, prop, ncp, output$est_afr_prop, output$est_afr_prop_rrss, w))

In [ ]:
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="AFR_EUR_hetero_v2"
job_dir="${base_dir}/${job_name}"
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_CAUSAL_NCP_PROP
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 90:00
#BSUB -o WORK_DIR/log/JOB_NAME_CAUSAL_NCP_PROP.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_CAUSAL_NCP_PROP.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/33.Fig4C_4D_S9B_S9D.AFR_EUR_Meta_v2.ipynb" afr_eur_hetero_v2 \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/kg_trimed_random/*.bim) \
    --n_causal CAUSAL --ncp NCP --afr_prop PROP --total_n 600 \
    --cwd WORK_DIR/result/ncp_NCP/causal_CAUSAL/prop_PROP/ \
    --mem 30G -j 4
EOF
chmod +x ${job_name}.sh

for causal in 1 2 3; do
  for ncp in 20 30 40; do
    for prop in 0.0 0.20 0.50 0.80 1.0; do
      output_script="${job_name}_causal_${causal}_ncp_${ncp}_prop_${prop}.sh"
      sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" \
          -e "s|CAUSAL|${causal}|g" \
          -e "s|NCP|${ncp}|g" \
          -e "s|PROP|${prop}|g" \
          ${job_name}.sh > "${output_script}"
      sb "${output_script}"
    done
  done
done

## Summary

`[meta_v2_summary]` from `32` on `AFR_EUR_meta_v2/result/`. The hetero outputs hold only the weight estimates (`est_afr_prop`, `est_afr_prop_rrss`, `w`) and are read directly by the data notebook.

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_meta_v2"
mkdir -p ${work_dir}/{summary,log}
cd ${work_dir}/summary

cat << 'EOF' > summary.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J AFR_EUR_meta_v2_summary
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/log/summary.%J.out
#BSUB -e WORK_DIR/log/summary.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/32.Fig4A_4B_S9A.ADSP_UKB_Meta_v2.ipynb" meta_v2_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary --mem 40G
EOF
sed "s|WORK_DIR|${work_dir}|g" summary.sh > summary_submit.sh
sb summary_submit.sh